# পাঠ ১৩ - Cognee জ্ঞানের গ্রাফ সহ এজেন্ট স্মৃতি


## সেটআপ

এই নোটবুকটি দেখায় কিভাবে [**Cognee**](https://www.cognee.ai/) জ্ঞান গ্রাফ এবং **Microsoft Agent Framework** (MAF) ব্যবহার করে একটি বুদ্ধিমান **কোডিং অ্যাসিস্ট্যান্ট** তৈরি করা যায় যা স্থায়ী স্মৃতি রাখে।

Cognee অগঠনমূলক টেক্সটকে একটি কাঠামোবদ্ধ, অনুসন্ধানযোগ্য জ্ঞান গ্রাফে রূপান্তরিত করে যা ভেক্টর এমবেডিং দ্বারা সমর্থিত — আপনার এজেন্টকে একটি সমৃদ্ধ, সম্পর্ক সচেতন দীর্ঘমেয়াদী স্মৃতি দেয়।

### আপনি যা শিখবেন
1. **জ্ঞান গ্রাফ নির্মাণ**: ডেভেলপার প্রোফাইল এবং সেরা অনুশীলনগুলি কাঠামোবদ্ধ, অনুসন্ধানযোগ্য জ্ঞান হিসাবে রূপান্তর করা।
2. **Cognee কে MAF এর সাথে সংযুক্ত করা**: `@tool` ফাংশন ব্যবহার করে MAF এজেন্টকে Cognee এর জ্ঞান গ্রাফ থেকে তথ্য অনুসন্ধান করতে দেয়া।
3. **সেশন-সচেতন আলাপচারিতা**: একই সেশনে একাধিক প্রশ্নের মধ্যে প্রাসঙ্গিকতা বজায় রাখা।
4. **দীর্ঘমেয়াদী স্মৃতি**: সেশন জুড়ে গুরুত্বপূর্ণ জ্ঞান সংরক্ষণ করা এবং নতুন কথোপকথনে তা পুনরুদ্ধার করা।

### পূর্বপ্রয়োজনীয়তাসমূহ
- পাইথন 3.9+
- লোকাল হোস্টে Redis রানিং (`docker run -d -p 6379:6379 redis`) সেশন ব্যবস্থাপনার জন্য
- একটি LLM API কী (যেমন OpenAI) — `.env` এ `LLM_API_KEY` সেট করুন
- `.env` এ `CACHING=true` (Cognee সেশনগুলির জন্য প্রয়োজন)
- একটি Microsoft Foundry প্রকল্প সহ একটি ডিপ্লয় করা চ্যাট মডেল
- `.env` এ `AZURE_AI_PROJECT_ENDPOINT` এবং `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI তে লগইন করা (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## এজেন্ট মেমোরির ধরনসমূহ

এই নোটবুকটি প্রধান পাঠ ১৩ নোটবুকের একই তিনটি মেমোরি ধরন অন্বেষণ করে, কিন্তু দীর্ঘমেয়াদি মেমোরি ব্যাকএন্ড হিসাবে Cognee ব্যবহার করে:

| মেমোরি ধরন | প্রক্রিয়া | জীবনকাল |
|---|---|---|
| **কার্যকরী** | `agent.create_session()` (MAF) | একক কথোপকথন |
| **স্বল্পমেয়াদি** | Cognee সেশন ক্যাশ (Redis) | একক সেশন |
| **দীর্ঘমেয়াদি** | Cognee জ্ঞান গ্রাফ + ভেক্টর | স্থায়ী |

### Cognee এর মেমোরি আর্কিটেকচার
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## কগনী স্টোরেজ প্রস্তুত করুন


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## অংশ ১ — জ্ঞানভিত্তি তৈরির কাজ

আমরা একটি পূর্ণাঙ্গ জ্ঞানভিত্তি তৈরির জন্য তিন ধরনের ডেটা গ্রহণ করি আমাদের কোডিং সহায়কের জন্য:

১. **ডেভেলপার প্রোফাইল** — ব্যক্তিগত দক্ষতা এবং প্রযুক্তিগত পটভূমি
২. **পাইথনের সেরা অভ্যাস** — পাইথনের জেন সাথে প্রায়োগিক নির্দেশিকা
৩. **ঐতিহাসিক আলোচনা** — ডেভেলপার এবং AI সহায়কদের মধ্যে পূর্ববর্তী প্রশ্নোত্তর সেশনসমূহ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## জ্ঞান গ্রাফ ভিজুয়ালাইজ করুন

Cognee এটি যে সত্তা এবং সম্পর্ক নির্ণয় করেছে তার একটি ইন্টারেক্টিভ HTML ভিজুয়ালাইজেশন প্রদর্শন করতে পারে।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## মেমরি সমৃদ্ধ করুন Memify দিয়ে

`memify()` জ্ঞান গ্রাফ বিশ্লেষণ করে এবং বুদ্ধিমত্তাসম্পন্ন নিয়মগুলি তৈরি করে — নিদর্শন, সর্বোত্তম অনুশীলন এবং ধারণাগুলির মধ্যে সম্পর্ক চিহ্নিত করে।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## অংশ ২ — Cognee টুলস সহ MAF এজেন্ট

এখন আমরা একটি MAF এজেন্ট তৈরি করব যা `@tool` ফাংশনের মাধ্যমে Cognee এর জ্ঞান গ্রাফকে জিজ্ঞাসা করতে পারে। এটি এজেন্টকে গ্রাফ-অবগত সেমান্টিক অনুসন্ধানের পূর্ণ ক্ষমতা ব্যবহার করতে দেয়, একাধিক সেশনের মাধ্যমে কথোপকথনের প্রসঙ্গ বজায় রাখার সময়। 


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## সেশন সহ ওয়ার্কিং মেমরি

`AgentSession` (যা `agent.create_session()` এর মাধ্যমে তৈরি) একটি সেশনের মধ্যে ওয়ার্কিং মেমরি প্রদান করে। এজেন্ট পূর্ববর্তী বার্তা গুলির দিকে ফিরে দেখতে পারে পাশাপাশি Cognee-এর দীর্ঘমেয়াদি জ্ঞানের গ্রাফেও অনুসন্ধান করতে পারে।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## নতুন সেশন — দীর্ঘমেয়াদী স্মৃতি টিকে থাকে

একটি নতুন সেশন শুরু করলে ওয়ার্কিং মেমোরি মুছে যায়, কিন্তু Cognee জ্ঞান গ্রাফ এখনও উপলব্ধ থাকে। এজেন্ট পুরোপুরি নতুন কথোপকথনে একই দীর্ঘমেয়াদী জ্ঞান পুনরুদ্ধার করতে পারে।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারমর্ম

এই নোটবুকে আপনি এমন একটি কোডিং সহকারী তৈরি করেছেন যা **MAF-এর ওয়ার্কিং মেমোরি** (`agent.create_session()`) ও **Cognee-এর দীর্ঘমেয়াদী জ্ঞান গ্রাফ** একসাথে সংযুক্ত করে।

### আপনি যা শিখেছেন
1. **জ্ঞান গ্রাফ নির্মাণ**: Cognee অবিন্যস্ত পাঠ গ্রহণ করে একটি গ্রাফ + ভেক্টর মেমোরি তৈরি করে।
2. **memify সহ গ্রাফ সমৃদ্ধকরণ**: আপনার বিদ্যমান গ্রাফের উপর থেকে প্রাপ্ত তথ্য ও সমৃদ্ধ সম্পর্ক।
3. **MAF + Cognee সমন্বয়**: `@tool` ফাংশনগুলি MAF এজেন্টকে Cognee-এর গ্রাফকে স্বাভাবিকভাবে অনুসন্ধান করতে দেয়।
4. **ওয়ার্কিং মেমোরি + দীর্ঘমেয়াদী মেমোরি**: `AgentSession` ( `agent.create_session()` এর মাধ্যমে) সেশন প্রসঙ্গ প্রদান করে, আর Cognee স্থায়ী জ্ঞান প্রদান করে।
5. **NodeSets সহ ফিল্টার করা অনুসন্ধান**: জ্ঞান গ্রাফের নির্দিষ্ট উপসেট লক্ষ্য করা (যেমন কেবল নীতি)।

### মূল বিষয়সমূহ
- **Cognee** কাঁচা টেক্সটকে গঠনমূলক, সম্পর্ক সচেতন মেমোরিতে রূপান্তর করে — একটি সমান্তরাল ভেক্টর স্টোরের থেকে শক্তিশালী।
- **`@tool` ফাংশনগুলি** MAF এজেন্ট এবং বহিরাগত জ্ঞান ব্যবস্থাগুলির মধ্যে পরিষ্কার সংযোগ স্থাপন করে।
- **`AgentSession`** ( `agent.create_session()` এর মাধ্যমে) প্রতিটি সংলাপের প্রসঙ্গকে দীর্ঘস্থায়ী জ্ঞান থেকে পৃথক রাখে।
- একই জ্ঞান গ্রাফ একাধিক সেশন ও এজেন্টের জন্য কাজ করে।

### বাস্তব বিশ্বের অ্যাপ্লিকেশনসমূহ
- **ডেভেলপার কপিলটস**: কোড পর্যালোচনা, ঘটনার বিশ্লেষণ, আর্কিটেকচার সহায়ক
- **গ্রাহক-সম্মুখ কপিলটস**: প্রোডাক্ট ডকুমেন্ট, FAQ, ও CRM নোটসের উপর ভিত্তি করে সাপোর্ট এজেন্ট
- **অভ্যন্তরীণ বিশেষজ্ঞ কপিলটস**: নীতিমালা, আইন বা নিরাপত্তা সহায়ক গাইডলাইন অনুযায়ী যুক্তি প্রদান করে
- **একীভূত তথ্য স্তর**: কাঠামোবদ্ধ ও অবিন্যস্ত ডেটাকে একত্রিত করে একটি অনুসন্ধানযোগ্য গ্রাফে রূপান্তর

### পরবর্তী ধাপগুলি
- Cognee-তে কালীন সচেতনতা নিয়ে পরীক্ষা নিরীক্ষা করুন
- ডোমেন-নির্দিষ্ট গ্রাফ গুণগত মানের জন্য একটি OWL ওন্টোলজি সংজ্ঞায়িত করুন
- সময়ে সময়ে পুনরুদ্ধারের উন্নতির জন্য ব্যবহারকারীর প্রতিক্রিয়া লুপ যোগ করুন
- একই Cognee মেমোরি স্তর ব্যবহার করে বহু-এজেন্ট সিস্টেমে স্কেল করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
